# Johdanto prompttien suunnitteluun
Prompttien suunnittelu on prosessi, jossa suunnitellaan ja optimoidaan kehotteita luonnollisen kielen käsittelytehtäviin. Siihen kuuluu oikeiden kehotteiden valinta, niiden parametrien virittäminen ja suorituskyvyn arviointi. Prompttien suunnittelu on ratkaisevan tärkeää saavutettaessa korkeaa tarkkuutta ja tehokkuutta NLP-malleissa. Tässä osiossa tutustumme prompttien suunnittelun perusteisiin käyttäen OpenAI:n malleja tutkimiseen.


### Harjoitus 1: Tokenisointi
Tutustu tokenisointiin käyttämällä tiktokenia, OpenAI:n avointa ja nopeaa tokenisoijaa.
Katso lisää esimerkkejä osoitteesta [OpenAI Cookbook](https://github.com/openai/openai-cookbook/blob/main/examples/How_to_count_tokens_with_tiktoken.ipynb?WT.mc_id=academic-105485-koreyst).


In [ ]:
# EXERCISE:
# 1. Run the exercise as is first
# 2. Change the text to any prompt input you want to use & re-run to see tokens

import tiktoken

# Define the prompt you want tokenized
text = f"""
Jupiter is the fifth planet from the Sun and the \
largest in the Solar System. It is a gas giant with \
a mass one-thousandth that of the Sun, but two-and-a-half \
times that of all the other planets in the Solar System combined. \
Jupiter is one of the brightest objects visible to the naked eye \
in the night sky, and has been known to ancient civilizations since \
before recorded history. It is named after the Roman god Jupiter.[19] \
When viewed from Earth, Jupiter can be bright enough for its reflected \
light to cast visible shadows,[20] and is on average the third-brightest \
natural object in the night sky after the Moon and Venus.
"""

# Set the model you want encoding for
encoding = tiktoken.encoding_for_model("gpt-4o")

# Encode the text - gives you the tokens in integer form
tokens = encoding.encode(text)
print(tokens);

# Decode the integers to see what the text versions look like
[encoding.decode_single_token_bytes(token) for token in tokens]


### Harjoitus 2: Vahvista Microsoft Foundry Models -avaimen asetus

> **Huom:** GitHub Models lopetetaan heinäkuun 2026 lopussa. Tämä harjoitus käyttää sen sijaan [Microsoft Foundry Models](https://ai.azure.com/catalog/models?WT.mc_id=academic-105485-koreyst) -palvelua, joka tarjoaa saman ilmaisen kokeiltavan malliluettelon ja Azure AI Inference SDK -kokemuksen.

Suorita alla oleva koodi varmistaaksesi, että Microsoft Foundry Models -päätepisteesi on asetettu oikein. Koodi yrittää yksinkertaisen peruskyselyn ja vahvistaa suorituksen. Syöte `oh say can you see` pitäisi täydentyä suunnilleen muotoon `by the dawn's early light..`


In [ ]:
import os
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import SystemMessage, UserMessage
from azure.core.credentials import AzureKeyCredential

# Get these from your Microsoft Foundry project's "Overview" page
token = os.environ["AZURE_INFERENCE_CREDENTIAL"]
endpoint = os.environ["AZURE_INFERENCE_ENDPOINT"]

model_name = "gpt-4o-mini"

client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)

def get_completion(prompt, client, model_name, temperature=1.0, max_tokens=1000, top_p=1.0):
    response = client.complete(
        messages=[
            {
                "role": "system",
                "content": "You are a helpful assistant.",
            },
            {
                "role": "user",
                "content": prompt,
            },
        ],
        model=model_name,
        temperature=temperature,
        max_tokens=max_tokens,
        top_p=top_p
    )
    return response.choices[0].message.content

## ---------- Call the helper method

### 1. Set primary content or prompt text
text = f"""
oh say can you see
"""

### 2. Use that in the prompt template below
prompt = f"""
```{text}```
"""

## 3. Run the prompt
response = get_completion(prompt, client, model_name)
print(response)


That line is the opening lyric of "The Star-Spangled Banner," the national anthem of the United States, written by Francis Scott Key. If you'd like more information or analysis, feel free to ask!


### Harjoitus 3: Keksinnöt
Tutki, mitä tapahtuu, kun pyydät LLM:ää palauttamaan täydennyksiä kehotteelle aiheesta, joka ei ehkä ole olemassa, tai aiheista, joista se ei ehkä tiedä, koska ne ovat olleet sen esikoulutusjoukon ulkopuolella (tuoreempia). Katso, miten vastaus muuttuu, jos kokeilet eri kehotetta tai eri mallia.


In [ ]:

## Set the text for simple prompt or primary content
## Prompt shows a template format with text in it - add cues, commands etc if needed
## Run the completion 
text = f"""
generate a lesson plan on the Martian War of 2076.
"""

prompt = f"""
```{text}```
"""

response = get_completion(prompt, client, model_name)
print(response)

### Harjoitus 4: Ohjeisiin Perustuva
Käytä muuttujaa "text" pääsisällön asettamiseen
ja muuttujaa "prompt" antaa siihen liittyvä ohje.

Tässä pyydämme mallia tiivistämään tekstin toisen luokan oppilaalle


In [4]:
# Test Example
# https://platform.openai.com/playground/p/default-summarize

## Example text
text = f"""
Jupiter is the fifth planet from the Sun and the \
largest in the Solar System. It is a gas giant with \
a mass one-thousandth that of the Sun, but two-and-a-half \
times that of all the other planets in the Solar System combined. \
Jupiter is one of the brightest objects visible to the naked eye \
in the night sky, and has been known to ancient civilizations since \
before recorded history. It is named after the Roman god Jupiter.[19] \
When viewed from Earth, Jupiter can be bright enough for its reflected \
light to cast visible shadows,[20] and is on average the third-brightest \
natural object in the night sky after the Moon and Venus.
"""

## Set the prompt
prompt = f"""
Summarize content you are provided with for a second-grade student.
```{text}```
"""

## Run the prompt
response = get_completion(prompt, client, model_name)
print(response)

Jupiter is the fifth planet from the Sun and the biggest one in our Solar System. It's made of gas and is much bigger than all the other planets put together! You can see Jupiter in the night sky because it's very bright. People have noticed it for a really long time and named it after a Roman god.


### Harjoitus 5: Monimutkainen pyyntö 
Kokeile pyyntöä, jossa on järjestelmän, käyttäjän ja avustajan viestejä 
Järjestelmä asettaa avustajan kontekstin
Käyttäjän ja avustajan viestit tarjoavat monivaiheisen keskustelukontekstin

Huomaa, kuinka avustajan persoona on asetettu "sarkastiseksi" järjestelmän kontekstissa. 
Kokeile käyttää eri persoonakontekstia. Tai kokeile erilaista syöte-/lähtöviestisarjaa


In [5]:
response = client.complete(
    model=model_name,
    messages=[
        {"role": "system", "content": "You are a sarcastic assistant."},
        {"role": "user", "content": "Who won the world series in 2020?"},
        {"role": "assistant", "content": "Who do you think won? The Los Angeles Dodgers of course."},
        {"role": "user", "content": "Where was it played?"}
    ]
)
print(response.choices[0].message.content)

Oh, you mean the famous 2020 World Series that wasn’t in a regular location? That was the year they played in the glamorous Arlington, Texas, at Globe Life Field.


### Harjoitus: Tutki intuitiotasi
Yllä olevat esimerkit antavat sinulle malleja, joita voit käyttää uusien kehotteiden luomiseen (yksinkertaisia, monimutkaisia, ohjeita jne.) – kokeile luoda muita harjoituksia tutkiaksesi joitakin muita ideoita, joista olemme puhuneet, kuten esimerkkejä, vihjeitä ja muuta.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Vastuuvapauslauseke**:
Tämä asiakirja on käännetty käyttämällä tekoälypohjaista käännöspalvelua [Co-op Translator](https://github.com/Azure/co-op-translator). Vaikka pyrimme tarkkuuteen, otathan huomioon, että automaattiset käännökset saattavat sisältää virheitä tai epätarkkuuksia. Alkuperäinen asiakirja sen alkuperäiskielellä on virallinen lähde. Tärkeissä asioissa suositellaan ammattimaista ihmiskäännöstä. Emme ole vastuussa tämän käännöksen käytöstä aiheutuvista väärinymmärryksistä tai tulkinnoista.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
